# TRELLIS.2 - Bild zu 3D für OpenTTS

**Microsoft's Image-to-3D mit vollen PBR-Texturen (MIT Lizenz)**

**Workflow:**
1. Setup ausführen (einmal, ~5 min)
2. Bild hochladen
3. 3D-Modell generieren
4. Herunterladen

**Wichtig:** Benötigt A100 GPU (Colab Pro) - mindestens 24GB VRAM!

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/DutchMaxwell/openTTS/blob/main/tools/3d-generation/trellis2_colab.ipynb)

---
## Schritt 1: Setup (einmal ausführen)
Dauert ca. 5 Minuten. Danach kannst du beliebig viele Bilder konvertieren.

In [ ]:
#@title 1. Setup ausführen {display-mode: "form"}

import torch
import os
import time
import sys
from IPython.display import display, HTML, clear_output

def progress_bar(step, total, desc=""):
    """Zeigt einen Fortschrittsbalken an"""
    percent = int((step / total) * 100)
    bar_length = 30
    filled = int(bar_length * step / total)
    bar = "█" * filled + "░" * (bar_length - filled)
    print(f"\r[{bar}] {percent}% - {desc}", end="", flush=True)
    if step == total:
        print()

def run_install(cmd, desc, step, total):
    """Führt Installation aus mit Fortschrittsanzeige"""
    progress_bar(step - 1, total, f"{desc}...")
    result = os.system(f"{cmd} > /dev/null 2>&1")
    progress_bar(step, total, f"{desc} ✓")
    return result == 0

# GPU Check
print("=" * 50)
print("GPU CHECK")
print("=" * 50)

if not torch.cuda.is_available():
    print("❌ KEINE GPU! Gehe zu: Runtime > Change runtime type > GPU")
    raise SystemExit("GPU required")

gpu_name = torch.cuda.get_device_name(0)
vram = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f"GPU: {gpu_name}")
print(f"VRAM: {vram:.1f} GB")

if vram < 23:
    print("\n❌ NICHT GENUG VRAM!")
    print("TRELLIS.2 benötigt mindestens 24GB (A100).")
    print("\nAlternativen:")
    print("1. Colab Pro mit A100 nutzen")
    print("2. HuggingFace Space: https://huggingface.co/spaces/microsoft/TRELLIS.2")
    raise SystemExit("Insufficient VRAM")
else:
    print(f"\n✅ GPU OK: {gpu_name}")

# Installation
if not os.path.exists('/content/TRELLIS.2/trellis2'):
    start_time = time.time()
    
    print("\n" + "=" * 50)
    print("INSTALLATION")
    print("=" * 50)
    print("⏳ Das dauert ca. 8-10 Minuten beim ersten Mal.\n")
    
    total_steps = 8
    
    # Step 1: Clone
    progress_bar(0, total_steps, "Repository klonen...")
    os.system("git clone --depth 1 https://github.com/microsoft/TRELLIS.2.git /content/TRELLIS.2 > /dev/null 2>&1")
    os.chdir("/content/TRELLIS.2")
    progress_bar(1, total_steps, "Repository klonen ✓")
    
    # Step 2: Core packages
    run_install("pip install -q imageio imageio-ffmpeg tqdm easydict opencv-python-headless", 
                "Core-Pakete (1/3)", 2, total_steps)
    
    # Step 3: More core packages
    run_install("pip install -q ninja trimesh transformers gradio tensorboard pandas lpips", 
                "Core-Pakete (2/3)", 3, total_steps)
    
    # Step 4: ML packages
    run_install("pip install -q zstandard kornia timm pillow huggingface_hub einops safetensors accelerate diffusers", 
                "Core-Pakete (3/3)", 4, total_steps)
    
    # Step 5: Flash attention (long!)
    print()
    print("\n⏳ Flash Attention kompilieren (ca. 5 min)...")
    os.system("pip install flash-attn==2.7.3 --no-build-isolation 2>/dev/null || true")
    progress_bar(5, total_steps, "Flash Attention ✓")
    
    # Step 6: nvdiffrast
    print()
    print("\n⏳ CUDA Renderer kompilieren (ca. 2 min)...")
    os.system("pip install git+https://github.com/NVlabs/nvdiffrast > /dev/null 2>&1")
    progress_bar(6, total_steps, "CUDA Renderer ✓")
    
    # Step 7: Submodules
    run_install("git submodule update --init --recursive", 
                "Extensions", 7, total_steps)
    
    # Step 8: Finalize
    sys.path.insert(0, '/content/TRELLIS.2')
    progress_bar(8, total_steps, "Fertig!")
    
    elapsed = time.time() - start_time
    print(f"\n\n✅ Installation abgeschlossen in {elapsed/60:.1f} Minuten!")
else:
    os.chdir("/content/TRELLIS.2")
    sys.path.insert(0, '/content/TRELLIS.2')
    print("\n✅ TRELLIS.2 bereits installiert")

# Create folders
os.makedirs('/content/input', exist_ok=True)
os.makedirs('/content/output', exist_ok=True)

print("\n" + "=" * 50)
print("✅ SETUP FERTIG - Weiter zu Schritt 2!")
print("=" * 50)

---
## Schritt 2: Bild hochladen

In [ ]:
#@title 2. Bild hochladen {display-mode: "form"}

from google.colab import files
from pathlib import Path
import shutil
from IPython.display import display, Image as IPImage

print("Wähle ein Bild aus (PNG oder JPG)...")
print("")

uploaded = files.upload()

if uploaded:
    for filename in uploaded.keys():
        dest = f'/content/input/{filename}'
        shutil.move(filename, dest)
        
        print(f"\n✅ Hochgeladen: {filename}")
        print("\nVorschau:")
        display(IPImage(dest, width=300))
        
        current_image = dest
        %store current_image
        
    print("\n" + "=" * 50)
    print("✅ BILD BEREIT - Weiter zu Schritt 3!")
    print("=" * 50)
else:
    print("❌ Kein Bild hochgeladen")

---
## Schritt 3: 3D-Modell generieren

In [ ]:
#@title 3. 3D-Modell generieren {display-mode: "form"}

#@markdown ### Einstellungen:
aufloesung = "512" #@param ["512", "1024", "1536"]

from pathlib import Path
from PIL import Image
import time
import sys
sys.path.insert(0, '/content/TRELLIS.2')

# Get uploaded image
%store -r current_image

try:
    img_path = Path(current_image)
except:
    images = list(Path('/content/input').glob('*.png')) + list(Path('/content/input').glob('*.jpg'))
    if images:
        img_path = images[-1]
    else:
        print("❌ Kein Bild gefunden! Führe zuerst Schritt 2 aus.")
        raise SystemExit()

output_path = f'/content/output/{img_path.stem}.glb'

print("=" * 50)
print(f"Eingabe: {img_path.name}")
print(f"Auflösung: {aufloesung}³")
print("=" * 50)

# Load pipeline
print("\n🔄 Lade TRELLIS.2 Modell...")
from trellis2.pipelines import Trellis2ImageTo3DPipeline

pipeline = Trellis2ImageTo3DPipeline.from_pretrained("microsoft/TRELLIS.2-4B")
pipeline.to("cuda")

# Load image
image = Image.open(img_path).convert("RGB")

# Generate
print(f"\n🔄 Generiere 3D-Modell...")
start = time.time()

mesh = pipeline.run(
    image,
    resolution=int(aufloesung)
)[0]

# Export
mesh.export(output_path)

elapsed = time.time() - start

if Path(output_path).exists():
    size_mb = Path(output_path).stat().st_size / 1e6
    print("\n" + "=" * 50)
    print(f"✅ FERTIG in {elapsed:.0f} Sekunden!")
    print(f"📦 Datei: {Path(output_path).name} ({size_mb:.1f} MB)")
    print("=" * 50)
    print("\nWeiter zu Schritt 4 zum Herunterladen!")
    
    current_output = output_path
    %store current_output
else:
    print("\n❌ Fehler bei der Generierung")

---
## Schritt 4: Herunterladen

In [ ]:
#@title 4. GLB-Datei herunterladen {display-mode: "form"}

from google.colab import files
from pathlib import Path

%store -r current_output

try:
    output_path = Path(current_output)
except:
    outputs = list(Path('/content/output').glob('*.glb'))
    if outputs:
        output_path = outputs[-1]
    else:
        print("❌ Kein 3D-Modell gefunden! Führe zuerst Schritt 3 aus.")
        raise SystemExit()

if output_path.exists():
    size_mb = output_path.stat().st_size / 1e6
    print(f"📦 Download: {output_path.name} ({size_mb:.1f} MB)")
    print("")
    files.download(str(output_path))
    print("\n" + "=" * 50)
    print("✅ Download gestartet!")
    print("")
    print("Nächste Schritte:")
    print("1. GLB-Datei in OpenTTS importieren")
    print("2. Spawn > Import GLB")
    print("3. Positionieren und skalieren")
    print("4. L drücken zum Fixieren")
    print("=" * 50)
else:
    print("❌ Datei nicht gefunden")

---
## Weitere Bilder?

Gehe zurück zu **Schritt 2** und lade das nächste Bild hoch.

Das Modell bleibt geladen - weitere Generierungen sind schneller!

---
## Kein A100? Nutze den HuggingFace Space!

Kostenlos im Browser: **[TRELLIS.2 Demo](https://huggingface.co/spaces/microsoft/TRELLIS.2)**